In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
import warnings


from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)

warnings.filterwarnings('ignore')


# import warnings
# warnings.filterwarnings('ignore')


In [ ]:

# Get all CSV files from the ../../data/machines folder
csv_files = glob.glob('../../data/azure_pm/machines/*.csv')

# Read each CSV file and create dataframes with the same name as the file
for file_path in csv_files:
    # Extract filename without extension
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    
    # Read CSV and assign to variable with the same name as the file
    globals()[file_name] = pd.read_csv(file_path)
    
    print(f"Loaded {file_name}.csv with shape: {globals()[file_name].shape}")

In [ ]:
# machine_1.head()
machine_1.head()

<br> <br>

## Data Encoding and Scaling

In [ ]:
def encode_and_scale_dataframe(df, target_column='failure', categorical_cols=['model'], numerical_cols=['age', 'volt', 'pressure', 'vibration'], scaling_method='standard', exclude_features=None, remove_excluded=False):
  
    # Handle exclude_features parameter
    if exclude_features is None:
        exclude_features = []
    elif isinstance(exclude_features, str):
        exclude_features = [exclude_features]
    
    # Create a copy to avoid modifying original dataframe
    df_processed = df.copy()
    
    print("🔄 Starting encoding and scaling process...")
    print(f"📊 Original dataframe shape: {df_processed.shape}")
    
    if exclude_features:
        print(f"🚫 Excluding features from processing: {exclude_features}")
        if remove_excluded:
            print(f"🗑️  Features will be removed from output dataframe")
    
    # Dictionary to store encoders for later use
    encoders = {}
    
    # 1. ENCODE CATEGORICAL VARIABLES
    print("\n1️⃣ ENCODING CATEGORICAL VARIABLES")
    print("-" * 40)
    
    # Filter out excluded features from categorical columns
    categorical_cols_to_process = [col for col in categorical_cols if col not in exclude_features]
    excluded_categorical = [col for col in categorical_cols if col in exclude_features]
    
    if excluded_categorical:
        print(f"   🚫 Skipping categorical encoding for: {excluded_categorical}")
    
    for col in categorical_cols_to_process:
        if col in df_processed.columns:
            print(f"   Encoding '{col}'...")
            
            # Create and fit label encoder
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col].astype(str))
            
            # Store encoder for future use
            encoders[col] = le
            
            # Display encoding mapping
            unique_values = df[col].unique()
            encoded_values = le.transform(unique_values.astype(str))
            mapping = dict(zip(unique_values, encoded_values))
            
            print(f"   ✅ {col} encoded: {mapping}")
        else:
            print(f"   ⚠️  Column '{col}' not found in dataframe")
    
    # 2. ENCODE TARGET VARIABLE (FAILURE)
    print(f"\n2️⃣ ENCODING TARGET VARIABLE: '{target_column}'")
    print("-" * 40)
    
    if target_column in exclude_features:
        print(f"   🚫 Skipping target variable encoding (excluded): '{target_column}'")
    elif target_column in df_processed.columns:
        print(f"   Encoding '{target_column}'...")
        
        # Check if target is already numeric
        if df_processed[target_column].dtype in ['object', 'category']:
            le_target = LabelEncoder()
            df_processed[target_column] = le_target.fit_transform(df_processed[target_column].astype(str))
            encoders[target_column] = le_target
            
            # Display target encoding mapping
            unique_targets = df[target_column].unique()
            encoded_targets = le_target.transform(unique_targets.astype(str))
            target_mapping = dict(zip(unique_targets, encoded_targets))
            print(f"   ✅ {target_column} encoded: {target_mapping}")
        else:
            print(f"   ℹ️  {target_column} is already numeric")
    else:
        print(f"   ⚠️  Target column '{target_column}' not found")
    
    # 3. SCALE NUMERICAL FEATURES
    print(f"\n3️⃣ SCALING NUMERICAL FEATURES ({scaling_method.upper()})")
    print("-" * 40)
    
    # Filter out excluded features from numerical columns
    numerical_cols_to_process = [col for col in numerical_cols if col not in exclude_features]
    excluded_numerical = [col for col in numerical_cols if col in exclude_features]
    
    if excluded_numerical:
        print(f"   🚫 Skipping scaling for: {excluded_numerical}")
    
    # Filter numerical columns that exist in dataframe
    existing_numerical_cols = [col for col in numerical_cols_to_process if col in df_processed.columns]
    missing_numerical_cols = [col for col in numerical_cols_to_process if col not in df_processed.columns]
    
    if missing_numerical_cols:
        print(f"   ⚠️  Missing columns: {missing_numerical_cols}")
    
    if existing_numerical_cols:
        print(f"   Scaling columns: {existing_numerical_cols}")
        
        # Choose scaler
        if scaling_method.lower() == 'standard':
            scaler = StandardScaler()
            print("   📏 Using StandardScaler (mean=0, std=1)")
        elif scaling_method.lower() == 'minmax':
            scaler = MinMaxScaler()
            print("   📏 Using MinMaxScaler (range 0-1)")
        else:
            scaler = StandardScaler()
            print("   📏 Default: Using StandardScaler")
        
        # Apply scaling
        df_processed[existing_numerical_cols] = scaler.fit_transform(df_processed[existing_numerical_cols])
        
        print("   ✅ Scaling completed")
        
        # Display scaling statistics
        print("\n   📊 SCALING STATISTICS:")
        for col in existing_numerical_cols:
            original_mean = df[col].mean()
            original_std = df[col].std()
            scaled_mean = df_processed[col].mean()
            scaled_std = df_processed[col].std()
            
            print(f"   {col:12} | Original: μ={original_mean:6.2f}, σ={original_std:6.2f} | "
                  f"Scaled: μ={scaled_mean:6.2f}, σ={scaled_std:6.2f}")
    else:
        scaler = None
        print("   ⚠️  No numerical columns to scale")
    
    # 4. REMOVE EXCLUDED FEATURES (if requested)
    if remove_excluded and exclude_features:
        print(f"\n🗑️  REMOVING EXCLUDED FEATURES")
        print("-" * 40)
        features_to_remove = [col for col in exclude_features if col in df_processed.columns]
        if features_to_remove:
            df_processed = df_processed.drop(columns=features_to_remove)
            print(f"   Removed columns: {features_to_remove}")
        else:
            print("   No excluded features found in dataframe")
    
    # 5. SUMMARY
    print(f"\n{'5️⃣' if remove_excluded else '4️⃣'} PROCESSING SUMMARY")
    print("-" * 40)
    print(f"   📊 Final dataframe shape: {df_processed.shape}")
    print(f"   🔤 Categorical columns encoded: {len([col for col in categorical_cols_to_process if col in encoders])}")
    print(f"   🔢 Numerical columns scaled: {len(existing_numerical_cols) if existing_numerical_cols else 0}")
    print(f"   🚫 Features excluded from processing: {len(exclude_features)}")
    if remove_excluded and exclude_features:
        print(f"   🗑️  Features removed from dataframe: {len([col for col in exclude_features if col in df.columns])}")
    print(f"   🎯 Target variable processed: {'Yes' if target_column in df_processed.columns and target_column not in exclude_features else 'No'}")
    
    return df_processed, encoders, scaler

In [ ]:
# Create a dictionary of all machine dataframes
machine_dataframes = {}

# Get a list of variable names first to avoid iteration issues
var_names = list(globals().keys())

for var_name in var_names:
    if var_name.startswith('machine_') and isinstance(globals()[var_name], pd.DataFrame):
        machine_dataframes[var_name] = globals()[var_name]

# Apply encoding and scaling to all machine dataframes
processed_machine_dataframes = {}
encoders_dict = {}
scalers_dict = {}

print(f"Processing {len(machine_dataframes)} machine dataframes...")
print("=" * 60)

for machine_key, machine_df in machine_dataframes.items():
    print(f"\n🔧 Processing {machine_key}...")
    
    # Apply encoding and scaling
    processed_df, encoders, scaler = encode_and_scale_dataframe(
        df=machine_df,
        target_column='failure',
        categorical_cols=['errorID', 'comp'],
        numerical_cols=['volt', 'rotate', 'pressure', 'vibration'],
        scaling_method='minmax',
        exclude_features=['machineID', "age", "model"],
        remove_excluded=True, 
    )

    
    # Store processed dataframe and encoders/scalers
    processed_machine_dataframes[machine_key] = processed_df
    encoders_dict[machine_key] = encoders
    scalers_dict[machine_key] = scaler
    
    print(f"✅ {machine_key} processed successfully - Shape: {processed_df.shape}")

print(f"\n🎉 All {len(processed_machine_dataframes)} machine dataframes processed!")
print(f"📊 Total processed dataframes: {len(processed_machine_dataframes)}")

In [ ]:
processed_machine_dataframes["machine_1"].head()

In [ ]:
processed_machine_dataframes["machine_1"]["errorID"].value_counts()

<br> <br> <br>

### Create lag features

In [ ]:
## Fixed Temporal Feature Engineering with Categorical Target

def create_lag_features(df, target_col='failure', prediction_horizon=24):
    """
    Create lag features and CATEGORICAL target for time series prediction with NO future data leakage.
    
    Args:
        df: Input dataframe sorted by datetime
        target_col: Column containing failure information (categorical)
        prediction_horizon: Hours ahead to predict (default: 24 hours)
    
    Returns:
        DataFrame with lag features and proper CATEGORICAL target variable
    """
    print(f"🔄 Creating lag features and CATEGORICAL target (predicting {prediction_horizon}h ahead)...")
    print(f"Original target distribution: {df[target_col].value_counts().to_dict()}")
    
    # Ensure data is sorted by time
    df_sorted = df.sort_values('datetime').reset_index(drop=True)
    
    # Keep failure as categorical (no conversion to binary)
    print(f"Categorical failure distribution: {df_sorted[target_col].value_counts().to_dict()}")
    print(f"Unique failure types: {sorted(df_sorted[target_col].unique())}")
    
    # Create target: predict failure type within next N hours using forward-looking window
    # For categorical data, we'll take the maximum failure type in the next N hours
    df_sorted['target'] = df_sorted[target_col].rolling(window=prediction_horizon, min_periods=1).max().shift(-prediction_horizon).fillna(0)
    
    # Ensure target maintains the same data type as original failure column
    df_sorted['target'] = df_sorted['target'].astype(df_sorted[target_col].dtype)
    
    display(df_sorted["target"].value_counts())

    # Remove last N rows where we can't predict the future
    df_sorted = df_sorted.iloc[:-prediction_horizon].copy()
    
    # Create lag features for sensor data (only use PAST data)
    sensor_cols = ['volt', 'rotate', 'pressure', 'vibration']
    
    print(f"Creating lag features for sensor columns: {sensor_cols}")
    
    for col in sensor_cols:
        # Lag features (1, 6, 12, 24 hours ago)
        for lag in [1, 6, 12, 24]:
            df_sorted[f'{col}_lag_{lag}h'] = df_sorted[col].shift(lag)
        
        # Rolling statistics (past 24 hours)
        df_sorted[f'{col}_mean_24h'] = df_sorted[col].rolling(window=24, min_periods=1).mean()
        df_sorted[f'{col}_std_24h'] = df_sorted[col].rolling(window=24, min_periods=1).std()
        df_sorted[f'{col}_min_24h'] = df_sorted[col].rolling(window=24, min_periods=1).min()
        df_sorted[f'{col}_max_24h'] = df_sorted[col].rolling(window=24, min_periods=1).max()
        
        # Rolling statistics (past 6 hours)
        df_sorted[f'{col}_mean_6h'] = df_sorted[col].rolling(window=6, min_periods=1).mean()
        df_sorted[f'{col}_std_6h'] = df_sorted[col].rolling(window=6, min_periods=1).std()
    
    # Error and maintenance lag features
    df_sorted['errorID_lag_1h'] = df_sorted['errorID'].shift(1)
    df_sorted['errorID_lag_6h'] = df_sorted['errorID'].shift(6)
    df_sorted['errorID_lag_12h'] = df_sorted['errorID'].shift(12)
    df_sorted['comp_lag_1h'] = df_sorted['comp'].shift(1)
    df_sorted['comp_lag_6h'] = df_sorted['comp'].shift(6)
    df_sorted['comp_lag_12h'] = df_sorted['comp'].shift(12)
    
    # Count features (past events only)
    df_sorted['error_count_6h'] = df_sorted['errorID'].rolling(window=6, min_periods=1).sum()
    df_sorted['error_count_24h'] = df_sorted['errorID'].rolling(window=24, min_periods=1).sum()
    df_sorted['maint_count_6h'] = df_sorted['comp'].rolling(window=6, min_periods=1).sum()
    df_sorted['maint_count_24h'] = df_sorted['comp'].rolling(window=24, min_periods=1).sum()
    
    # Time features
    df_sorted['hour'] = df_sorted['datetime'].dt.hour
    df_sorted['day_of_week'] = df_sorted['datetime'].dt.dayofweek
    df_sorted['is_weekend'] = (df_sorted['datetime'].dt.dayofweek >= 5).astype(int)
    df_sorted['is_working_hours'] = ((df_sorted['hour'] >= 8) & (df_sorted['hour'] <= 17)).astype(int)
    
    # Hours since last maintenance
    maint_mask = df_sorted['comp'] > 0
    if maint_mask.any():
        last_maint_idx = -1
        hours_since_maint = []
        for i, is_maint in enumerate(maint_mask):
            if is_maint:
                last_maint_idx = i
                hours_since_maint.append(0)
            else:
                hours_since_maint.append(i - last_maint_idx if last_maint_idx >= 0 else i)
        df_sorted['hours_since_maint'] = hours_since_maint
    else:
        df_sorted['hours_since_maint'] = range(len(df_sorted))
    
    # Hours since last error
    error_mask = df_sorted['errorID'] > 0
    if error_mask.any():
        last_error_idx = -1
        hours_since_error = []
        for i, is_error in enumerate(error_mask):
            if is_error:
                last_error_idx = i
                hours_since_error.append(0)
            else:
                hours_since_error.append(i - last_error_idx if last_error_idx >= 0 else i)
        df_sorted['hours_since_error'] = hours_since_error
    else:
        df_sorted['hours_since_error'] = range(len(df_sorted))
    
    # Fill NaN values created by lag features
    df_sorted = df_sorted.fillna(0)
    
    print(f"✅ Created lag features. Final shape: {df_sorted.shape}")
    print(f"   CATEGORICAL Target distribution: {df_sorted['target'].value_counts().to_dict()}")
    print(f"   Target classes: {sorted(df_sorted['target'].unique())}")
    print(f"   Target data type: {df_sorted['target'].dtype}")
    print(f"   New feature count: {len([col for col in df_sorted.columns if col not in df.columns])}")
    
    return df_sorted


In [ ]:
# Apply lag feature engineering to all processed machine dataframes
lag_machine_dataframes = {}

print(f"Creating lag features for {len(processed_machine_dataframes)} machine dataframes...")
print("=" * 70)

for machine_key, processed_df in processed_machine_dataframes.items():
    print(f"\n🔧 Processing {machine_key}...")
    
    # Get the original machine dataframe to access datetime column
    original_df = machine_dataframes[machine_key].copy()
    
    # Add datetime back to processed dataframe for lag feature creation
    processed_df_with_datetime = processed_df.copy()
    processed_df_with_datetime['datetime'] = original_df['datetime'].values
    
    # Convert datetime column to proper datetime format
    processed_df_with_datetime['datetime'] = pd.to_datetime(processed_df_with_datetime['datetime'])
    
    # Apply lag feature engineering
    df_with_lags = create_lag_features(
        df=processed_df_with_datetime,
        target_col='failure',
        prediction_horizon=24
    )
    
    # Store the result
    lag_machine_dataframes[machine_key] = df_with_lags
    
    print(f"✅ {machine_key} lag features created - Shape: {df_with_lags.shape}")

print(f"\n🎉 All {len(lag_machine_dataframes)} machine dataframes processed with lag features!")
print(f"📊 Total lag dataframes created: {len(lag_machine_dataframes)}")

# Display sample from one machine
sample_machine = 'machine_100'
if sample_machine in lag_machine_dataframes:
    print(f"\n📋 Sample from {sample_machine}:")
    print(f"Columns ({len(lag_machine_dataframes[sample_machine].columns)}): {list(lag_machine_dataframes[sample_machine].columns)}")
    print(f"Shape: {lag_machine_dataframes[sample_machine].shape}")

<br> <br> <br>

#### Save lag features to ../../data/azure_pm/machines/lag_features

In [ ]:
# Save lag_machine_dataframes to ../../data/azure_pm/machines/lag_features

# Create the output directory if it doesn't exist
output_dir = '../../data/azure_pm/machines/lag_features'
os.makedirs(output_dir, exist_ok=True)

print(f"💾 Saving {len(lag_machine_dataframes)} lag feature dataframes...")
print("=" * 60)

# Save each lag feature dataframe
for machine_key, lag_df in lag_machine_dataframes.items():
    output_path = os.path.join(output_dir, f'{machine_key}_lag_features.csv')
    
    # Save to CSV
    lag_df.to_csv(output_path, index=False)
    
    print(f"✅ Saved {machine_key} - Shape: {lag_df.shape} - File: {output_path}")

print(f"\n🎉 All lag feature dataframes saved successfully!")
print(f"📁 Output directory: {output_dir}")
print(f"📊 Total files saved: {len(lag_machine_dataframes)}")

# Display directory contents
saved_files = [f for f in os.listdir(output_dir) if f.endswith('_lag_features.csv')]
print(f"\n📋 Saved files ({len(saved_files)}):")
for i, filename in enumerate(sorted(saved_files)[:10]):  # Show first 10
    print(f"  {i+1:2d}. {filename}")
if len(saved_files) > 10:
    print(f"  ... and {len(saved_files) - 10} more files")

In [ ]:


# # Save lag_machine_dataframes to ../../data/azure_pm/machines

# # Create the output directory if it doesn't exist
# output_dir = '../../data/azure_pm/lag_features'
# os.makedirs(output_dir, exist_ok=True)

# print(f"💾 Saving {len(lag_machine_dataframes)} lag feature dataframes...")
# print("=" * 60)

# # Save each lag feature dataframe
# for machine_key, lag_df in lag_machine_dataframes.items():
#     output_path = os.path.join(output_dir, f'{machine_key}_lag_features.csv')
    
#     # Save to CSV
#     lag_df.to_csv(output_path, index=False)
    
#     print(f"✅ Saved {machine_key} - Shape: {lag_df.shape} - File: {output_path}")

# print(f"\n🎉 All lag feature dataframes saved successfully!")
# print(f"📁 Output directory: {output_dir}")
# print(f"📊 Total files saved: {len(lag_machine_dataframes)}")

# # Display directory contents
# saved_files = [f for f in os.listdir(output_dir) if f.endswith('_lag_features.csv')]
# print(f"\n📋 Saved files ({len(saved_files)}):")
# for i, filename in enumerate(sorted(saved_files)[:10]):  # Show first 10
#     print(f"  {i+1:2d}. {filename}")
# if len(saved_files) > 10:
#     print(f"  ... and {len(saved_files) - 10} more files")


In [ ]:
lag_machine_dataframes["machine_1"]["target"].value_counts()

In [ ]:
def walk_forward_validation(df, n_splits=5):
    """
    Perform walk-forward validation with expanding window for MULTICLASS/CATEGORICAL classification.
    
    Args:
        df: DataFrame with features and CATEGORICAL target
        n_splits: Number of validation splits
        
    Returns:
        Dictionary with results for each model
    """
    print(f"🚀 Starting Walk-Forward Validation with {n_splits} splits...")
    print("=" * 60)
    
    # Prepare features and target
    exclude_cols = ['datetime', 'machineID', 'failure', 'binary_failure', 'target']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    X = df[feature_cols].fillna(0)  # Fill any remaining NaN
    y_original = df['target']
    
    # FIXED: Remap target classes to be consecutive for XGBoost compatibility
    unique_classes = sorted(y_original.unique())
    n_classes = len(unique_classes)
    
    # Create mapping from original classes to consecutive classes
    class_mapping = {original_class: consecutive_class for consecutive_class, original_class in enumerate(unique_classes)}
    reverse_mapping = {consecutive_class: original_class for original_class, consecutive_class in class_mapping.items()}
    
    # Apply mapping to target
    y = y_original.map(class_mapping)
    
    print(f"📊 Dataset info:")
    print(f"   - Total samples: {len(X):,}")
    print(f"   - Features: {len(feature_cols)}")
    print(f"   - Original target classes: {unique_classes}")
    print(f"   - Remapped target classes: {sorted(y.unique())}")
    print(f"   - Class mapping: {class_mapping}")
    print(f"   - Number of classes: {n_classes}")
    print(f"   - Target distribution: {y.value_counts().to_dict()}")
    
    if n_classes < 2:
        print(f"❌ ERROR: Need at least 2 classes for classification, but found: {n_classes}")
        return None, None
    
    # Determine if binary or multiclass
    is_binary = n_classes == 2 and set(unique_classes) == {0, 1}
    classification_type = "BINARY" if is_binary else "MULTICLASS"
    
    print(f"🎯 Classification type: {classification_type}")
    
    # Model configurations
    models = {
        'RandomForest': RandomForestClassifier(
            n_estimators=100, max_depth=10, min_samples_split=5,
            random_state=42, n_jobs=-1
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, verbose=-1, force_col_wise=True
        ),
        'XGBoost': XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, eval_metric='logloss', verbosity=0
        ),
        'CatBoost': CatBoostClassifier(
            iterations=100, depth=6, learning_rate=0.1,
            random_state=42, verbose=False, allow_writing_files=False
        )
    }
    
    # Enhanced results storage - Modified for multiclass
    results = {name: {
        'auc': [],  # Will be macro-averaged AUC for multiclass
        'precision': [], 'recall': [], 'accuracy': [], 
        'f1_score': [],  # NEW: F1 score (important for multiclass)
        'macro_precision': [],  # NEW: Macro-averaged precision
        'macro_recall': [],  # NEW: Macro-averaged recall
        'weighted_precision': [],  # NEW: Weighted precision
        'weighted_recall': [],  # NEW: Weighted recall
        'per_class_metrics': [],  # NEW: Per-class precision, recall, f1
        'confusion_matrices': [],  # NEW: Confusion matrices
        'predictions': [], 'y_true': [], 'pred_probabilities': []  # Modified
    } for name in models.keys()}
    
    # For binary classification, keep the enhanced ROC metrics
    if is_binary:
        for name in results.keys():
            results[name].update({
                'normalized_auc': [],
                'optimal_threshold': [],
                'optimal_precision': [],
                'optimal_recall': [],
                'youden_index': [],
                'roc_curves': []
            })
    
    # Create time series splits (expanding window)
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    fold = 1
    for train_idx, test_idx in tscv.split(X):
        print(f"\n📅 Fold {fold}/{n_splits}")
        print("-" * 30)
        
        # Split data
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Check classes in training and test sets
        train_classes = sorted(y_train.unique())
        test_classes = sorted(y_test.unique())
        
        if len(train_classes) < 2:
            print(f"⚠️  Skipping fold {fold}: Training set has only one class: {train_classes}")
            fold += 1
            continue
        
        print(f"   Train: {len(X_train):,} samples, classes: {train_classes}")
        print(f"   Test:  {len(X_test):,} samples, classes: {test_classes}")
        print(f"   Train distribution: {y_train.value_counts().to_dict()}")
        print(f"   Test distribution: {y_test.value_counts().to_dict()}")
        
        # Calculate class weights for imbalanced data
        try:
            class_weights = compute_class_weight(
                'balanced', classes=np.unique(y_train), y=y_train
            )
            class_weight_dict = {cls: weight for cls, weight in zip(np.unique(y_train), class_weights)}
            
            # For XGBoost multiclass, we'll use class weights differently
            if is_binary and len(class_weights) > 1:
                scale_pos_weight = class_weights[1] / class_weights[0]
            else:
                scale_pos_weight = 1
        except:
            class_weight_dict = None
            scale_pos_weight = 1
        
        # Train each model
        for model_name, base_model in models.items():
            try:
                # Configure model with class weights
                if model_name == 'RandomForest':
                    model = RandomForestClassifier(
                        n_estimators=100, max_depth=10, min_samples_split=5,
                        class_weight='balanced', random_state=42, n_jobs=-1
                    )
                elif model_name == 'LightGBM':
                    model = LGBMClassifier(
                        n_estimators=100, max_depth=6, learning_rate=0.1,
                        class_weight='balanced', random_state=42, 
                        verbose=-1, force_col_wise=True
                    )
                elif model_name == 'XGBoost':
                    if is_binary:
                        model = XGBClassifier(
                            n_estimators=100, max_depth=6, learning_rate=0.1,
                            scale_pos_weight=scale_pos_weight, random_state=42,
                            eval_metric='logloss', verbosity=0
                        )
                    else:
                        # For multiclass, XGBoost automatically handles it
                        # FIXED: Ensure classes are consecutive
                        model = XGBClassifier(
                            n_estimators=100, max_depth=6, learning_rate=0.1,
                            random_state=42, eval_metric='mlogloss', verbosity=0, 
                            num_classes=n_classes
                        )
                elif model_name == 'CatBoost':
                    if is_binary:
                        model = CatBoostClassifier(
                            iterations=100, depth=6, learning_rate=0.1,
                            class_weights=[1, scale_pos_weight], random_state=42,
                            verbose=False, allow_writing_files=False
                        )
                    else:
                        # For multiclass, use auto class weights
                        model = CatBoostClassifier(
                            iterations=100, depth=6, learning_rate=0.1,
                            auto_class_weights='Balanced', random_state=42,
                            verbose=False, allow_writing_files=False
                        )
                
                # Train model
                model.fit(X_train, y_train)
                
                # Make predictions
                y_pred = model.predict(X_test)
                y_pred_proba = model.predict_proba(X_test)
                
                # Calculate metrics based on classification type
                if is_binary:
                    # Binary classification metrics (existing logic)
                    y_pred_proba_pos = y_pred_proba[:, 1]  # Positive class probabilities
                    
                    try:
                        if len(y_test.unique()) > 1:
                            # Standard AUC
                            auc = roc_auc_score(y_test, y_pred_proba_pos)
                            
                            # Normalized AUC Score
                            normalized_auc = (auc - 0.5) / 0.5
                            
                            # Calculate ROC curve and Youden's index
                            fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_pos)
                            
                            # Calculate Youden's index
                            youden_index = tpr - fpr
                            optimal_idx = np.argmax(youden_index)
                            optimal_threshold = thresholds[optimal_idx]
                            max_youden_index = youden_index[optimal_idx]
                            
                            # Calculate precision and recall at optimal threshold
                            y_pred_optimal = (y_pred_proba_pos >= optimal_threshold).astype(int)
                            optimal_precision = precision_score(y_test, y_pred_optimal, zero_division=0)
                            optimal_recall = recall_score(y_test, y_pred_optimal, zero_division=0)
                            
                            # Store ROC curve data
                            roc_data = {'fpr': fpr, 'tpr': tpr, 'thresholds': thresholds, 
                                      'optimal_idx': optimal_idx}
                        else:
                            auc = 0.5
                            normalized_auc = 0.0
                            optimal_threshold = 0.5
                            max_youden_index = 0.0
                            optimal_precision = 0.0
                            optimal_recall = 0.0
                            roc_data = None
                    except:
                        auc = 0.5
                        normalized_auc = 0.0
                        optimal_threshold = 0.5
                        max_youden_index = 0.0
                        optimal_precision = 0.0
                        optimal_recall = 0.0
                        roc_data = None
                    
                    # Store binary-specific metrics
                    results[model_name]['normalized_auc'].append(normalized_auc)
                    results[model_name]['optimal_threshold'].append(optimal_threshold)
                    results[model_name]['optimal_precision'].append(optimal_precision)
                    results[model_name]['optimal_recall'].append(optimal_recall)
                    results[model_name]['youden_index'].append(max_youden_index)
                    results[model_name]['roc_curves'].append(roc_data)
                    
                else:
                    # Multiclass classification metrics
                    try:
                        # Macro-averaged AUC (one-vs-rest)
                        auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
                    except:
                        auc = 0.5
                
                # Standard metrics (work for both binary and multiclass)
                precision = precision_score(y_test, y_pred, zero_division=0, average='macro')
                recall = recall_score(y_test, y_pred, zero_division=0, average='macro')
                accuracy = accuracy_score(y_test, y_pred)
                
                # NEW: Additional multiclass metrics
                f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
                macro_precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
                macro_recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
                weighted_precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
                weighted_recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
                
                # Per-class metrics
                per_class_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
                
                # Confusion matrix
                conf_matrix = confusion_matrix(y_test, y_pred)
                
                # Store all results
                results[model_name]['auc'].append(auc)
                results[model_name]['precision'].append(precision)
                results[model_name]['recall'].append(recall)
                results[model_name]['accuracy'].append(accuracy)
                results[model_name]['f1_score'].append(f1)
                results[model_name]['macro_precision'].append(macro_precision)
                results[model_name]['macro_recall'].append(macro_recall)
                results[model_name]['weighted_precision'].append(weighted_precision)
                results[model_name]['weighted_recall'].append(weighted_recall)
                results[model_name]['per_class_metrics'].append(per_class_report)
                results[model_name]['confusion_matrices'].append(conf_matrix)
                results[model_name]['predictions'].append(y_pred)
                results[model_name]['y_true'].append(y_test)
                results[model_name]['pred_probabilities'].append(y_pred_proba)
                
                # Output metrics
                if is_binary:
                    print(f"   {model_name:12}: AUC={auc:.3f}, Norm_AUC={normalized_auc:.3f}, "
                          f"Prec={precision:.3f}, Rec={recall:.3f}, Acc={accuracy:.3f}")
                    print(f"   {'':12}  Opt_Thresh={optimal_threshold:.3f}, "
                          f"Opt_Prec={optimal_precision:.3f}, Opt_Rec={optimal_recall:.3f}, "
                          f"Youden={max_youden_index:.3f}")
                else:
                    print(f"   {model_name:12}: AUC={auc:.3f}, F1={f1:.3f}, "
                          f"Prec={precision:.3f}, Rec={recall:.3f}, Acc={accuracy:.3f}")
                    print(f"   {'':12}  W_Prec={weighted_precision:.3f}, "
                          f"W_Rec={weighted_recall:.3f}")
                
            except Exception as e:
                print(f"   ❌ {model_name} failed: {str(e)}")
        
        fold += 1
    
    # Return results with class mapping information
    return results, feature_cols, class_mapping, reverse_mapping

In [ ]:

# Create main directory for models
main_model_dir = '../../model/azure_pm'
os.makedirs(main_model_dir, exist_ok=True)

# Initialize results storage - UPDATED for categorical classification
all_results = {}
model_performance = {
    'RandomForest': {
        'auc': [], 'precision': [], 'recall': [], 'accuracy': [],
        'f1_score': [], 'macro_precision': [], 'macro_recall': [],
        'weighted_precision': [], 'weighted_recall': []
    },
    'LightGBM': {
        'auc': [], 'precision': [], 'recall': [], 'accuracy': [],
        'f1_score': [], 'macro_precision': [], 'macro_recall': [],
        'weighted_precision': [], 'weighted_recall': []
    },
    'XGBoost': {
        'auc': [], 'precision': [], 'recall': [], 'accuracy': [],
        'f1_score': [], 'macro_precision': [], 'macro_recall': [],
        'weighted_precision': [], 'weighted_recall': []
    },
    'CatBoost': {
        'auc': [], 'precision': [], 'recall': [], 'accuracy': [],
        'f1_score': [], 'macro_precision': [], 'macro_recall': [],
        'weighted_precision': [], 'weighted_recall': []
    }
}

print(f"🚀 Running walk-forward validation for {len(lag_machine_dataframes)} machines...")
print("=" * 80)

# Process each machine
for machine_key, df_with_lags in lag_machine_dataframes.items():
    print(f"\n🔧 Processing {machine_key}...")
    print("-" * 50)
    
    # Create directory for this machine
    machine_dir = os.path.join(main_model_dir, machine_key)
    os.makedirs(machine_dir, exist_ok=True)
    
    try:
        # Run walk-forward validation - UPDATED to receive class mappings
        validation_results, feature_names, class_mapping, reverse_mapping = walk_forward_validation(df_with_lags, n_splits=3)
        
        if validation_results is None:
            print(f"❌ Skipping {machine_key}: validation failed")
            continue
        
        # Store results for this machine
        all_results[machine_key] = validation_results
        
        # Calculate mean metrics for this machine - UPDATED for categorical
        machine_metrics = {}
        for model_name, results in validation_results.items():
            if results['auc']:  # Check if we have results
                mean_auc = np.mean(results['auc'])
                mean_precision = np.mean(results['precision'])
                mean_recall = np.mean(results['recall'])
                mean_accuracy = np.mean(results['accuracy'])
                mean_f1 = np.mean(results['f1_score'])
                mean_macro_precision = np.mean(results['macro_precision'])
                mean_macro_recall = np.mean(results['macro_recall'])
                mean_weighted_precision = np.mean(results['weighted_precision'])
                mean_weighted_recall = np.mean(results['weighted_recall'])
                
                machine_metrics[model_name] = {
                    'auc': mean_auc,
                    'precision': mean_precision,
                    'recall': mean_recall,
                    'accuracy': mean_accuracy,
                    'f1_score': mean_f1,
                    'macro_precision': mean_macro_precision,
                    'macro_recall': mean_macro_recall,
                    'weighted_precision': mean_weighted_precision,
                    'weighted_recall': mean_weighted_recall
                }
                
                # Add to global performance tracking - UPDATED
                model_performance[model_name]['auc'].append(mean_auc)
                model_performance[model_name]['precision'].append(mean_precision)
                model_performance[model_name]['recall'].append(mean_recall)
                model_performance[model_name]['accuracy'].append(mean_accuracy)
                model_performance[model_name]['f1_score'].append(mean_f1)
                model_performance[model_name]['macro_precision'].append(mean_macro_precision)
                model_performance[model_name]['macro_recall'].append(mean_macro_recall)
                model_performance[model_name]['weighted_precision'].append(mean_weighted_precision)
                model_performance[model_name]['weighted_recall'].append(mean_weighted_recall)
                
                print(f"   {model_name:12}: AUC={mean_auc:.3f}, F1={mean_f1:.3f}, Prec={mean_precision:.3f}, Rec={mean_recall:.3f}, Acc={mean_accuracy:.3f}")
        
        # Train final models on full dataset and save
        print(f"   💾 Training and saving final models for {machine_key}...")
        
        # Prepare data for final training
        exclude_cols = ['datetime', 'machineID', 'failure', 'binary_failure', 'target']
        feature_cols = [col for col in df_with_lags.columns if col not in exclude_cols]
        
        X_full = df_with_lags[feature_cols].fillna(0)
        y_full_original = df_with_lags['target']
        
        # FIXED: Apply the same class mapping for final training
        y_full = y_full_original.map(class_mapping)
        
        # Check if we have multiple classes - UPDATED for categorical
        unique_classes_original = sorted(y_full_original.unique())
        unique_classes_mapped = sorted(y_full.unique())
        n_classes = len(unique_classes_original)
        
        if n_classes < 2:
            print(f"   ⚠️  Skipping model training: only one class in target")
            continue
        
        # Determine if binary or multiclass
        is_binary = n_classes == 2 and set(unique_classes_original) == {0, 1}
        
        print(f"   📊 Target info: {n_classes} classes, Binary: {is_binary}")
        print(f"   📊 Original class distribution: {y_full_original.value_counts().to_dict()}")
        print(f"   📊 Mapped class distribution: {y_full.value_counts().to_dict()}")
        
        # Train and save each model - UPDATED for categorical
        try:
            # Calculate class weights
            class_weights = compute_class_weight('balanced', classes=np.unique(y_full), y=y_full)
            
            # For binary classification
            if is_binary and len(class_weights) > 1:
                scale_pos_weight = class_weights[1] / class_weights[0]
            else:
                scale_pos_weight = 1
            
            # RandomForest
            rf_model = RandomForestClassifier(
                n_estimators=100, max_depth=10, min_samples_split=5,
                class_weight='balanced', random_state=42, n_jobs=-1
            )
            rf_model.fit(X_full, y_full)
            
            with open(os.path.join(machine_dir, 'RandomForest.pkl'), 'wb') as f:
                pickle.dump({
                    'model': rf_model,
                    'feature_names': feature_cols,
                    'metrics': machine_metrics.get('RandomForest', {}),
                    'encoders': encoders_dict.get(machine_key, {}),
                    'scaler': scalers_dict.get(machine_key),
                    'n_classes': n_classes,
                    'is_binary': is_binary,
                    'class_names': unique_classes_original,
                    'class_mapping': class_mapping,
                    'reverse_mapping': reverse_mapping
                }, f)
            
            # LightGBM
            lgb_model = LGBMClassifier(
                n_estimators=100, max_depth=6, learning_rate=0.1,
                class_weight='balanced', random_state=42, 
                verbose=-1, force_col_wise=True
            )
            lgb_model.fit(X_full, y_full)
            
            with open(os.path.join(machine_dir, 'LightGBM.pkl'), 'wb') as f:
                pickle.dump({
                    'model': lgb_model,
                    'feature_names': feature_cols,
                    'metrics': machine_metrics.get('LightGBM', {}),
                    'encoders': encoders_dict.get(machine_key, {}),
                    'scaler': scalers_dict.get(machine_key),
                    'n_classes': n_classes,
                    'is_binary': is_binary,
                    'class_names': unique_classes_original,
                    'class_mapping': class_mapping,
                    'reverse_mapping': reverse_mapping
                }, f)
            
            # XGBoost - UPDATED for multiclass with consecutive classes
            if is_binary:
                xgb_model = XGBClassifier(
                    n_estimators=100, max_depth=6, learning_rate=0.1,
                    scale_pos_weight=scale_pos_weight, random_state=42,
                    eval_metric='logloss', verbosity=0
                )
            else:
                # For multiclass classification - now uses consecutive classes
                xgb_model = XGBClassifier(
                    n_estimators=100, max_depth=6, learning_rate=0.1,
                    random_state=42, eval_metric='mlogloss', verbosity=0
                )
            
            xgb_model.fit(X_full, y_full)
            
            with open(os.path.join(machine_dir, 'XGBoost.pkl'), 'wb') as f:
                pickle.dump({
                    'model': xgb_model,
                    'feature_names': feature_cols,
                    'metrics': machine_metrics.get('XGBoost', {}),
                    'encoders': encoders_dict.get(machine_key, {}),
                    'scaler': scalers_dict.get(machine_key),
                    'n_classes': n_classes,
                    'is_binary': is_binary,
                    'class_names': unique_classes_original,
                    'class_mapping': class_mapping,
                    'reverse_mapping': reverse_mapping
                }, f)
            
            # CatBoost - UPDATED for multiclass
            if is_binary:
                cat_model = CatBoostClassifier(
                    iterations=100, depth=6, learning_rate=0.1,
                    class_weights=[1, scale_pos_weight], random_state=42,
                    verbose=False, allow_writing_files=False
                )
            else:
                # For multiclass classification
                cat_model = CatBoostClassifier(
                    iterations=100, depth=6, learning_rate=0.1,
                    auto_class_weights='Balanced', random_state=42,
                    verbose=False, allow_writing_files=False
                )
            
            cat_model.fit(X_full, y_full)
            
            with open(os.path.join(machine_dir, 'CatBoost.pkl'), 'wb') as f:
                pickle.dump({
                    'model': cat_model,
                    'feature_names': feature_cols,
                    'metrics': machine_metrics.get('CatBoost', {}),
                    'encoders': encoders_dict.get(machine_key, {}),
                    'scaler': scalers_dict.get(machine_key),
                    'n_classes': n_classes,
                    'is_binary': is_binary,
                    'class_names': unique_classes_original,
                    'class_mapping': class_mapping,
                    'reverse_mapping': reverse_mapping
                }, f)
            
            print(f"   ✅ Successfully saved 4 models for {machine_key}")
            
        except Exception as e:
            print(f"   ❌ Error training/saving models for {machine_key}: {str(e)}")
    
    except Exception as e:
        print(f"❌ Error processing {machine_key}: {str(e)}")
        continue

print(f"\n🎉 Processing completed!")
print(f"📁 Models saved in: {main_model_dir}")

# Calculate and display holistic report - UPDATED for categorical
print("\n" + "="*80)
print("📊 HOLISTIC PERFORMANCE REPORT - CATEGORICAL CLASSIFICATION")
print("="*80)

# Create summary dataframe - UPDATED with new metrics
summary_data = []
for model_name, metrics in model_performance.items():
    if metrics['auc']:  # Check if we have data
        summary_data.append({
            'Model': model_name,
            'Machines_Processed': len(metrics['auc']),
            'Mean_AUC': np.mean(metrics['auc']),
            'Std_AUC': np.std(metrics['auc']),
            'Mean_F1_Score': np.mean(metrics['f1_score']),
            'Std_F1_Score': np.std(metrics['f1_score']),
            'Mean_Precision': np.mean(metrics['precision']),
            'Std_Precision': np.std(metrics['precision']),
            'Mean_Recall': np.mean(metrics['recall']),
            'Std_Recall': np.std(metrics['recall']),
            'Mean_Accuracy': np.mean(metrics['accuracy']),
            'Std_Accuracy': np.std(metrics['accuracy']),
            'Mean_Macro_Precision': np.mean(metrics['macro_precision']),
            'Mean_Macro_Recall': np.mean(metrics['macro_recall']),
            'Mean_Weighted_Precision': np.mean(metrics['weighted_precision']),
            'Mean_Weighted_Recall': np.mean(metrics['weighted_recall']),
            'Min_AUC': np.min(metrics['auc']),
            'Max_AUC': np.max(metrics['auc']),
            'Min_F1': np.min(metrics['f1_score']),
            'Max_F1': np.max(metrics['f1_score'])
        })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    
    print("\n🏆 OVERALL MODEL PERFORMANCE SUMMARY")
    print("-" * 60)
    for _, row in summary_df.iterrows():
        print(f"\n{row['Model']}:")
        print(f"  Machines Processed: {row['Machines_Processed']}")
        print(f"  AUC: {row['Mean_AUC']:.3f} ± {row['Std_AUC']:.3f} (range: {row['Min_AUC']:.3f} - {row['Max_AUC']:.3f})")
        print(f"  F1 Score: {row['Mean_F1_Score']:.3f} ± {row['Std_F1_Score']:.3f} (range: {row['Min_F1']:.3f} - {row['Max_F1']:.3f})")
        print(f"  Precision (Macro): {row['Mean_Precision']:.3f} ± {row['Std_Precision']:.3f}")
        print(f"  Recall (Macro): {row['Mean_Recall']:.3f} ± {row['Std_Recall']:.3f}")
        print(f"  Accuracy: {row['Mean_Accuracy']:.3f} ± {row['Std_Accuracy']:.3f}")
        print(f"  Macro Precision: {row['Mean_Macro_Precision']:.3f}")
        print(f"  Macro Recall: {row['Mean_Macro_Recall']:.3f}")
        print(f"  Weighted Precision: {row['Mean_Weighted_Precision']:.3f}")
        print(f"  Weighted Recall: {row['Mean_Weighted_Recall']:.3f}")
    
    # Rank models by different metrics - UPDATED
    print("\n🥇 MODEL RANKINGS")
    print("-" * 40)
    
    # Rank by F1 Score (most important for multiclass)
    summary_df_f1 = summary_df.sort_values('Mean_F1_Score', ascending=False)
    print("\n📊 BY F1 SCORE:")
    for i, (_, row) in enumerate(summary_df_f1.iterrows(), 1):
        print(f"{i}. {row['Model']:12} - F1: {row['Mean_F1_Score']:.3f}")
    
    # Rank by AUC
    summary_df_auc = summary_df.sort_values('Mean_AUC', ascending=False)
    print("\n📊 BY AUC:")
    for i, (_, row) in enumerate(summary_df_auc.iterrows(), 1):
        print(f"{i}. {row['Model']:12} - AUC: {row['Mean_AUC']:.3f}")
    
    # Rank by Accuracy
    summary_df_acc = summary_df.sort_values('Mean_Accuracy', ascending=False)
    print("\n📊 BY ACCURACY:")
    for i, (_, row) in enumerate(summary_df_acc.iterrows(), 1):
        print(f"{i}. {row['Model']:12} - Acc: {row['Mean_Accuracy']:.3f}")
    
    # Save summary report
    summary_df.to_csv(os.path.join(main_model_dir, 'categorical_performance_report.csv'), index=False)
    print(f"\n💾 Summary report saved to: {os.path.join(main_model_dir, 'categorical_performance_report.csv')}")
    
    # Display the summary dataframe
    print("\n📋 DETAILED PERFORMANCE TABLE")
    print("-" * 100)
    # Display key columns
    display_cols = ['Model', 'Machines_Processed', 'Mean_AUC', 'Mean_F1_Score', 
                   'Mean_Precision', 'Mean_Recall', 'Mean_Accuracy']
    print(summary_df[display_cols].round(3).to_string(index=False))
    
    # Create visualization for categorical results
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Categorical Classification Performance Comparison', fontsize=16)
    
    # AUC comparison
    summary_df.plot(x='Model', y='Mean_AUC', kind='bar', ax=axes[0,0], 
                   color='skyblue', yerr=summary_df['Std_AUC'])
    axes[0,0].set_title('AUC Score Comparison')
    axes[0,0].set_ylabel('AUC')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # F1 Score comparison
    summary_df.plot(x='Model', y='Mean_F1_Score', kind='bar', ax=axes[0,1], 
                   color='lightcoral', yerr=summary_df['Std_F1_Score'])
    axes[0,1].set_title('F1 Score Comparison')
    axes[0,1].set_ylabel('F1 Score')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # Precision comparison
    summary_df.plot(x='Model', y='Mean_Precision', kind='bar', ax=axes[1,0], 
                   color='lightgreen', yerr=summary_df['Std_Precision'])
    axes[1,0].set_title('Precision Comparison (Macro)')
    axes[1,0].set_ylabel('Precision')
    axes[1,0].tick_params(axis='x', rotation=45)
    
    # Recall comparison
    summary_df.plot(x='Model', y='Mean_Recall', kind='bar', ax=axes[1,1], 
                   color='gold', yerr=summary_df['Std_Recall'])
    axes[1,1].set_title('Recall Comparison (Macro)')
    axes[1,1].set_ylabel('Recall')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ No performance data available to generate report")

print(f"\n📁 Total directories created: {len([d for d in os.listdir(main_model_dir) if os.path.isdir(os.path.join(main_model_dir, d))])}")
print(f"📁 Models directory structure:")
for machine_dir in sorted(os.listdir(main_model_dir)):
    machine_path = os.path.join(main_model_dir, machine_dir)
    if os.path.isdir(machine_path):
        pkl_files = [f for f in os.listdir(machine_path) if f.endswith('.pkl')]
        print(f"  {machine_dir}: {len(pkl_files)} models")



And here's an updated analysis function for the categorical results:



In [ ]:
def analyze_results_categorical(results, is_binary=False):
    """Analyze and visualize walk-forward validation results for categorical classification"""
    
    classification_type = "BINARY" if is_binary else "MULTICLASS"
    
    print("\n" + "="*80)
    print(f"📊 {classification_type} WALK-FORWARD VALIDATION RESULTS SUMMARY")
    print("="*80)
    
    # Calculate average metrics
    summary_data = []
    for model_name, metrics in results.items():
        if len(metrics['auc']) > 0:  # Only if we have results
            avg_metrics = {
                'Model': model_name,
                'AUC': np.mean(metrics['auc']),
                'AUC_std': np.std(metrics['auc']),
                'Precision': np.mean(metrics['precision']),
                'Precision_std': np.std(metrics['precision']),
                'Recall': np.mean(metrics['recall']),
                'Recall_std': np.std(metrics['recall']),
                'Accuracy': np.mean(metrics['accuracy']),
                'Accuracy_std': np.std(metrics['accuracy']),
                'F1_Score': np.mean(metrics['f1_score']),
                'F1_std': np.std(metrics['f1_score']),
                'Macro_Precision': np.mean(metrics['macro_precision']),
                'Macro_Recall': np.mean(metrics['macro_recall']),
                'Weighted_Precision': np.mean(metrics['weighted_precision']),
                'Weighted_Recall': np.mean(metrics['weighted_recall'])
            }
            
            # Add binary-specific metrics if available
            if is_binary and 'normalized_auc' in metrics:
                avg_metrics.update({
                    'Normalized_AUC': np.mean(metrics['normalized_auc']),
                    'Normalized_AUC_std': np.std(metrics['normalized_auc']),
                    'Optimal_Threshold': np.mean(metrics['optimal_threshold']),
                    'Optimal_Precision': np.mean(metrics['optimal_precision']),
                    'Optimal_Recall': np.mean(metrics['optimal_recall']),
                    'Youden_Index': np.mean(metrics['youden_index'])
                })
            
            summary_data.append(avg_metrics)
            
            print(f"\n{model_name.upper()}:")
            print(f"  AUC:              {avg_metrics['AUC']:.4f} ± {avg_metrics['AUC_std']:.4f}")
            print(f"  F1 SCORE:         {avg_metrics['F1_Score']:.4f} ± {avg_metrics['F1_std']:.4f}")
            print(f"  PRECISION:        {avg_metrics['Precision']:.4f} ± {avg_metrics['Precision_std']:.4f}")
            print(f"  RECALL:           {avg_metrics['Recall']:.4f} ± {avg_metrics['Recall_std']:.4f}")
            print(f"  ACCURACY:         {avg_metrics['Accuracy']:.4f} ± {avg_metrics['Accuracy_std']:.4f}")
            print(f"  MACRO PRECISION:  {avg_metrics['Macro_Precision']:.4f}")
            print(f"  MACRO RECALL:     {avg_metrics['Macro_Recall']:.4f}")
            print(f"  WEIGHTED PREC:    {avg_metrics['Weighted_Precision']:.4f}")
            print(f"  WEIGHTED RECALL:  {avg_metrics['Weighted_Recall']:.4f}")
            
            if is_binary and 'Normalized_AUC' in avg_metrics:
                print(f"  NORMALIZED AUC:   {avg_metrics['Normalized_AUC']:.4f} ± {avg_metrics['Normalized_AUC_std']:.4f}")
                print(f"  OPTIMAL THRESH:   {avg_metrics['Optimal_Threshold']:.4f}")
                print(f"  YOUDEN INDEX:     {avg_metrics['Youden_Index']:.4f}")
    
    # Create summary DataFrame
    summary_df = pd.DataFrame(summary_data)
    
    if len(summary_df) > 0:
        # Find best models by different metrics
        best_auc = summary_df.loc[summary_df['AUC'].idxmax(), 'Model']
        best_f1 = summary_df.loc[summary_df['F1_Score'].idxmax(), 'Model']
        best_accuracy = summary_df.loc[summary_df['Accuracy'].idxmax(), 'Model']
        
        print(f"\n🏆 BEST PERFORMING MODELS:")
        print(f"   Best AUC:          {best_auc} ({summary_df['AUC'].max():.4f})")
        print(f"   Best F1 Score:     {best_f1} ({summary_df['F1_Score'].max():.4f})")
        print(f"   Best Accuracy:     {best_accuracy} ({summary_df['Accuracy'].max():.4f})")
        
        # Visualization
        n_plots = 6 if is_binary else 4
        n_cols = 3 if is_binary else 2
        n_rows = 2
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 10))
        if n_cols == 2:
            axes = axes.reshape(n_rows, n_cols)
        
        fig.suptitle(f'{classification_type} Walk-Forward Validation Results Comparison', fontsize=16)
        
        # AUC comparison
        summary_df.plot(x='Model', y='AUC', kind='bar', ax=axes[0,0], color='skyblue', yerr=summary_df['AUC_std'])
        axes[0,0].set_title('AUC Score Comparison')
        axes[0,0].set_ylabel('AUC')
        axes[0,0].tick_params(axis='x', rotation=45)
        
        # F1 Score comparison
        summary_df.plot(x='Model', y='F1_Score', kind='bar', ax=axes[0,1], color='lightcoral', yerr=summary_df['F1_std'])
        axes[0,1].set_title('F1 Score Comparison')
        axes[0,1].set_ylabel('F1 Score')
        axes[0,1].tick_params(axis='x', rotation=45)
        
        # Precision comparison
        summary_df.plot(x='Model', y='Precision', kind='bar', ax=axes[1,0], color='lightgreen', yerr=summary_df['Precision_std'])
        axes[1,0].set_title('Precision Comparison')
        axes[1,0].set_ylabel('Precision')
        axes[1,0].tick_params(axis='x', rotation=45)
        
        # Recall comparison
        summary_df.plot(x='Model', y='Recall', kind='bar', ax=axes[1,1], color='gold', yerr=summary_df['Recall_std'])
        axes[1,1].set_title('Recall Comparison')
        axes[1,1].set_ylabel('Recall')
        axes[1,1].tick_params(axis='x', rotation=45)
        
        # Binary-specific plots
        if is_binary and axes.shape[1] > 2:
            # Normalized AUC
            summary_df.plot(x='Model', y='Normalized_AUC', kind='bar', ax=axes[0,2], color='purple', yerr=summary_df['Normalized_AUC_std'])
            axes[0,2].set_title('Normalized AUC Comparison')
            axes[0,2].set_ylabel('Normalized AUC')
            axes[0,2].tick_params(axis='x', rotation=45)
            
            # Youden Index
            summary_df.plot(x='Model', y='Youden_Index', kind='bar', ax=axes[1,2], color='orange')
            axes[1,2].set_title('Youden Index Comparison')
            axes[1,2].set_ylabel('Youden Index')
            axes[1,2].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        # Display confusion matrix for best model (last fold)
        best_model_name = summary_df.loc[summary_df['F1_Score'].idxmax(), 'Model']
        if 'confusion_matrices' in results[best_model_name] and len(results[best_model_name]['confusion_matrices']) > 0:
            last_cm = results[best_model_name]['confusion_matrices'][-1]
            
            plt.figure(figsize=(8, 6))
            import seaborn as sns
            sns.heatmap(last_cm, annot=True, fmt='d', cmap='Blues')
            plt.title(f'Confusion Matrix - {best_model_name} (Last Fold)')
            plt.ylabel('True Label')
            plt.xlabel('Predicted Label')
            plt.show()
    
    return summary_df

# Usage - Use the correct variable name
summary = analyze_results_categorical(model_performance, is_binary=False)




## 🎯 **Key Changes Made:**

1. **Automatic Detection**: Detects whether target is binary or multiclass
2. **Multiclass Metrics**: Added F1-score, macro/weighted precision/recall
3. **Per-Class Metrics**: Stores detailed classification reports
4. **Confusion Matrices**: Tracks confusion matrices for each fold
5. **Model Configuration**: Adjusts XGBoost and CatBoost for multiclass
6. **AUC Calculation**: Uses macro-averaged one-vs-rest AUC for multiclass
7. **Enhanced Visualization**: Includes F1-score and confusion matrix plots
8. **Flexible Output**: Adapts print statements based on classification type

The function now automatically handles both binary and multiclass scenarios, providing appropriate metrics for each case!